<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session4_Rigorous_Zero_Counting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install python-flint==0.9.0

In [8]:
#Initialize rigorous arthmetic
from flint import acb, acb_series, arb, ctx
from fractions import Fraction

# Working precision: 160 binary bits (~48 decimal digits)
ctx.prec = 160

print("Python-FLINT initialized")
print("Working precision:", ctx.prec, "bits")

# Test a Taylor-series evaluation of zeta.
z = acb("1/4", "12")

jet = acb_series([z, 1], prec=2).zeta()
coefficients = jet.coeffs()

print("\nTest point:", z)
print("Zeta value:", coefficients[0])
print("Zeta derivative:", coefficients[1])

Python-FLINT initialized
Working precision: 160 bits

Test point: 0.25000000000000000000000000000000000000000000000 + 12.000000000000000000000000000000000000000000000j
Zeta value: [1.0525479734936550019363906903970684298518084544 +/- 2.06e-47] + [-0.88806309278175408994143388234459167306068221181 +/- 2.09e-48]j
Zeta derivative: [-0.17538998224444725657594636887830881617848795395 +/- 4.41e-48] + [0.62109900367470495670393081990088161427625488952 +/- 4.70e-48]j


In [9]:
#Certify contour has no zeros
def fraction_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


def interval_ball(a, b):
    midpoint = (a + b) / 2
    radius = abs(b - a) / 2

    return arb(
        f"{midpoint.numerator}/{midpoint.denominator}",
        f"{radius.numerator}/{radius.denominator}"
    )


def segment_box(p, q):
    """
    A rectangular ball containing the entire straight segment
    from p to q.
    """
    real_box = interval_ball(p[0], q[0])
    imag_box = interval_ball(p[1], q[1])

    return acb(real_box, imag_box)


# Exact rational coordinates for the rectangle
corners = [
    (Fraction(1, 10), Fraction(10)),
    (Fraction(9, 10), Fraction(10)),
    (Fraction(9, 10), Fraction(60)),
    (Fraction(1, 10), Fraction(60)),
    (Fraction(1, 10), Fraction(10))
]

MAX_DEPTH = 24

stats = {
    "certified_pieces": 0,
    "max_depth": 0,
    "lower_bounds": [],
    "unresolved": []
}


def certify_segment(p, q, depth=0):

    stats["max_depth"] = max(stats["max_depth"], depth)

    box = segment_box(p, q)

    # Rigorous enclosure of zeta over this entire box
    zeta_box = box.zeta()

    # Certified lower bound for |zeta|
    lower = zeta_box.abs_lower()

    if lower.is_finite() and not lower.contains(0):
        stats["certified_pieces"] += 1
        stats["lower_bounds"].append(lower)
        return True

    # The enclosure is too wide. Subdivide the segment.
    if depth >= MAX_DEPTH:
        stats["unresolved"].append((p, q))
        return False

    midpoint = (
        (p[0] + q[0]) / 2,
        (p[1] + q[1]) / 2
    )

    left_ok = certify_segment(p, midpoint, depth + 1)
    right_ok = certify_segment(midpoint, q, depth + 1)

    return left_ok and right_ok


boundary_certified = True

print("CERTIFYING THE FOUR CONTOUR SIDES")
print("--------------------------------")

for i in range(4):
    success = certify_segment(corners[i], corners[i + 1])

    print(
        f"Side {i + 1}:",
        "CERTIFIED" if success else "NOT YET CERTIFIED"
    )

    boundary_certified = boundary_certified and success

print("\nBoundary zero-free:", boundary_certified)
print("Certified subsegments:", stats["certified_pieces"])
print("Maximum subdivision depth:", stats["max_depth"])
print("Unresolved segments:", len(stats["unresolved"]))

CERTIFYING THE FOUR CONTOUR SIDES
--------------------------------
Side 1: CERTIFIED
Side 2: CERTIFIED
Side 3: CERTIFIED
Side 4: CERTIFIED

Boundary zero-free: True
Certified subsegments: 1705
Maximum subdivision depth: 12
Unresolved segments: 0


In [10]:
#Integerate the logarathimic derivative
if not boundary_certified:
    raise RuntimeError(
        "Boundary certification failed. "
        "Do not proceed to the contour integral yet."
    )


def zeta_log_derivative(z, analytic):
    """
    Compute zeta'(z) / zeta(z) using a rigorous Taylor series.

    The integrand is meromorphic, so the analytic flag
    does not need special branch-cut handling.
    """

    jet = acb_series([z, 1], prec=2).zeta()
    coeff = jet.coeffs()

    zeta_value = coeff[0]
    zeta_derivative = coeff[1]

    return zeta_derivative / zeta_value


# Counterclockwise contour
tol = arb("1e-20")

side_integrals = []

print("CALCULATING CONTOUR INTEGRALS")
print("-----------------------------")

for i in range(4):

    start = acb(
        fraction_ball(corners[i][0]),
        fraction_ball(corners[i][1])
    )

    end = acb(
        fraction_ball(corners[i + 1][0]),
        fraction_ball(corners[i + 1][1])
    )

    result = acb.integral(
        zeta_log_derivative,
        start,
        end,
        abs_tol=tol,
        rel_tol=tol,
        deg_limit=80,
        eval_limit=200000,
        depth_limit=5000,
        use_heap=True
    )

    if not result.is_finite():
        raise RuntimeError(
            f"Side {i + 1}: integration did not produce a finite enclosure."
        )

    side_integrals.append(result)

    print(f"\nSide {i + 1} integral:")
    print(result)


total_integral = sum(side_integrals, acb(0))

print("\nTOTAL CONTOUR INTEGRAL")
print("----------------------")
print(total_integral)

CALCULATING CONTOUR INTEGRALS
-----------------------------

Side 1 integral:
[-0.185823134060090923613 +/- 4.06e-22] + [-0.016452708905442263308 +/- 2.04e-22]j

Side 2 integral:
[-1.0532614002903135092 +/- 8.50e-20] + [0.3214837217392086920 +/- 5.28e-20]j

Side 3 integral:
[0.902585331599631189128 +/- 7.27e-22] + [0.310728231964789428693 +/- 5.74e-22]j

Side 4 integral:
[+/- 1.64e+2] + [+/- 2.49e+2]j

TOTAL CONTOUR INTEGRAL
----------------------
[+/- 1.64e+2] + [+/- 2.49e+2]j


In [11]:
#Extract the certified integral
two_pi_i = 2 * acb.pi() * acb(0, 1)

zero_count_ball = total_integral / two_pi_i

unique_count = zero_count_ball.unique_fmpz()

print("ZERO COUNT ENCLOSURE")
print("--------------------")
print(zero_count_ball)

print("\nUNIQUE INTEGER CHECK")
print("--------------------")

if unique_count is None:
    print("NOT YET CERTIFIED")
    print("The enclosure does not isolate a unique integer.")

else:
    certified_count = int(unique_count)

    print("CERTIFIED ZERO COUNT:", certified_count)
    print("Matches Session 03:", certified_count == 13)

ZERO COUNT ENCLOSURE
--------------------
[+/- 39.7] + [+/- 26.1]j

UNIQUE INTEGER CHECK
--------------------
NOT YET CERTIFIED
The enclosure does not isolate a unique integer.
